# A/Bテストで効果を検証する（実践編・総合演習）

対照実験（A/Bテスト）のデータを題材に、効果があったかを統計的に正しく判定し、効果量まで示し、なぜランダム化で因果が言えるのかを押さえて、結論を図で伝えます。
統計分析・因果推論・データ可視化を1つの意思決定に向けてつなぐ総合演習です。

このノートブックは Web ページ（practice-ab-test.html）と同じコードを、上から順に実行できるようにまとめたものです。データはノートブック内でシミュレーション生成するため、外部データのダウンロードは不要です。

制作: Yuta Kimura（大阪公立大学工業高等専門学校）. 内容は分野標準の一般的なもので、特定の研究や論文には紐づきません。

In [ ]:
!pip install scipy statsmodels pandas matplotlib

## Step 1. データ入手：真の効果を仕込んだ実験データ

NumPy の乱数で、二値のコンバージョン（真の率 A=12%, B=15%）と連続値の滞在時間（真の平均 A=60秒, B=64秒, 標準偏差20秒）を、各群 2000 サンプルずつ生成します。乱数の種を固定して再現できるようにします。

In [ ]:
import numpy as np

rng = np.random.default_rng(42)  # 種を固定して再現可能に

n_a, n_b = 2000, 2000            # 各群のサンプルサイズ

# 真のコンバージョン率（B群に +3ポイントの効果を仕込む）
p_a, p_b = 0.12, 0.15
conv_a = rng.binomial(1, p_a, n_a)   # 0/1 の配列
conv_b = rng.binomial(1, p_b, n_b)

# 真の滞在時間（B群の平均を +4秒。標準偏差は共通20秒）
dwell_a = rng.normal(60, 20, n_a)
dwell_b = rng.normal(64, 20, n_b)

print("A: CV数", conv_a.sum(), " B: CV数", conv_b.sum())
print("A: 実測CV率", round(conv_a.mean(), 4),
      " B: 実測CV率", round(conv_b.mean(), 4))

## Step 2. 探索・可視化：まず目で確かめる

pandas で群別に集計し、matplotlib で分布を描きます。

In [ ]:
import pandas as pd

# 1行=1ユーザーの長い表にまとめる
df = pd.DataFrame({
    "group": ["A"] * n_a + ["B"] * n_b,
    "converted": np.concatenate([conv_a, conv_b]),
    "dwell": np.concatenate([dwell_a, dwell_b]),
})

summary = df.groupby("group").agg(
    n=("converted", "size"),
    conv_rate=("converted", "mean"),
    dwell_mean=("dwell", "mean"),
    dwell_sd=("dwell", "std"),
)
print(summary.round(3))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# 左: 滞在時間の分布を群ごとに重ねる
ax[0].hist(dwell_a, bins=30, alpha=0.6, label="A", color="#B7C0BB")
ax[0].hist(dwell_b, bins=30, alpha=0.6, label="B", color="#0E6F5C")
ax[0].set_xlabel("dwell time (s)"); ax[0].set_ylabel("count")
ax[0].set_title("Dwell time distribution"); ax[0].legend()

# 右: コンバージョン率の棒
rates = df.groupby("group")["converted"].mean()
ax[1].bar(rates.index, rates.values, color=["#B7C0BB", "#0E6F5C"])
ax[1].set_ylabel("conversion rate")
ax[1].set_title("Conversion rate by group")

plt.tight_layout(); plt.show()

## Step 3. 仮説検定：偶然では説明しにくいか

帰無仮説「2群に差はない」のもとで、観測された差以上に極端な差が出る確率が p 値です。二値はカイ二乗検定 `chi2_contingency`（等価な Z 検定 `proportions_ztest`）、連続値は Welch の2標本 t 検定 `ttest_ind` を使います。差の95%信頼区間も併せて求めます。

z = (p̂_B − p̂_A) / sqrt( p̂(1−p̂)(1/n_A + 1/n_B) )。p̂ は2群をまとめたプール推定率。

In [ ]:
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import (
    proportions_ztest, confint_proportions_2indep)

# 分割表: 行=群, 列=[非CV, CV]
table = [[n_a - conv_a.sum(), conv_a.sum()],
         [n_b - conv_b.sum(), conv_b.sum()]]
chi2, p_chi, dof, expected = chi2_contingency(table)
print(f"chi2 = {chi2:.3f}, p = {p_chi:.4f}")

# 等価なZ検定と、差(p_B - p_A)の95%信頼区間
count = np.array([conv_b.sum(), conv_a.sum()])
nobs = np.array([n_b, n_a])
z, p_z = proportions_ztest(count, nobs)
low, high = confint_proportions_2indep(
    conv_b.sum(), n_b, conv_a.sum(), n_a, method="wald")
print(f"z = {z:.3f}, p = {p_z:.4f}")
print(f"95% CI for (p_B - p_A) = [{low:.4f}, {high:.4f}]")

In [ ]:
from scipy.stats import ttest_ind

# Welch の2標本t検定（等分散を仮定しない）
t, p_t = ttest_ind(dwell_b, dwell_a, equal_var=False)
print(f"t = {t:.3f}, p = {p_t:.3e}")
print(f"平均差 = {dwell_b.mean() - dwell_a.mean():.3f} 秒")

## Step 4. 効果量：どれだけ大きい差か

p 値は差の大きさを語りません。連続値は Cohen's d（標準偏差を単位にした平均差）、二値は絶対差とリフト（相対変化）で効果量を示します。

d = (x̄_B − x̄_A) / s_pooled。

In [ ]:
def cohens_d(x, y):
    nx, ny = len(x), len(y)
    # プール標準偏差（不偏分散を自由度で重み付け）
    vx, vy = x.var(ddof=1), y.var(ddof=1)
    pooled_sd = np.sqrt(((nx - 1) * vx + (ny - 1) * vy) / (nx + ny - 2))
    return (x.mean() - y.mean()) / pooled_sd

d = cohens_d(dwell_b, dwell_a)
print(f"Cohen's d = {d:.3f}  （滞在時間の効果量）")

# 二値: 絶対差とリフト（相対変化）
rate_a, rate_b = conv_a.mean(), conv_b.mean()
abs_diff = rate_b - rate_a
lift = abs_diff / rate_a
print(f"絶対差 = {abs_diff:.4f}  リフト = {lift * 100:.1f}%")

## Step 5. 因果の考え方：なぜランダム化で因果が言えるのか

ランダム化割り当ては、既知・未知を問わずあらゆる背景要因（交絡）を平均的に両群へ均等に散らし、群間の差を施策の因果効果として読めるようにします。

割り当てや集計が健全かは A/Aテスト で点検します。同じ母集団を2分割し、本来差がないデータで検定して、有意になりにくいことを確かめます。のぞき見（peeking）や多重比較は第一種の誤りを膨らませるので、サンプルサイズと期間を事前に固定します。

In [ ]:
# 同じ真の率から2群を作り、差がないことを確認する（A/Aテスト）
aa1 = rng.binomial(1, p_a, n_a)
aa2 = rng.binomial(1, p_a, n_a)
table_aa = [[n_a - aa1.sum(), aa1.sum()],
            [n_a - aa2.sum(), aa2.sum()]]
chi2_aa, p_aa, _, _ = chi2_contingency(table_aa)
print(f"A/A: chi2 = {chi2_aa:.3f}, p = {p_aa:.3f}")
# 健全なら p は大きく、有意にならないのが普通

## Step 6. 結論と可視化：意思決定に落とす

点推定に95%信頼区間をエラーバーで重ねた図で結論をまとめます。区間が重ならず離れているほど、差の確からしさが伝わります。

In [ ]:
# 各群のコンバージョン率と、その95%信頼区間（Wald近似）
groups = ["A", "B"]
rates = np.array([conv_a.mean(), conv_b.mean()])
ns = np.array([n_a, n_b])
se = np.sqrt(rates * (1 - rates) / ns)
ci = 1.96 * se  # 95%区間の半幅

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(groups, rates, yerr=ci, capsize=10,
       color=["#B7C0BB", "#0E6F5C"])
ax.set_ylabel("conversion rate")
ax.set_title("A/B test: conversion rate (95% CI)")
for i, r in enumerate(rates):
    ax.text(i, r + ci[i] + 0.004, f"{r:.3f}", ha="center")
plt.tight_layout(); plt.show()

## まとめ

統計分析（帰無仮説・p値・信頼区間）、因果推論（ランダム化が交絡を断つ）、データ可視化（効果量と信頼区間の図）を1本のパイプラインにつなぎ、対照実験のデータから施策の効果を検証しました。

`p_a` や `p_b`、サンプルサイズを変えて上から再実行すると、有意にならなくなる境目が見えてきます。因果検証をさらに深めたい場合は、安井翔太『効果検証入門』などが実務に即した入り口になります。